# **Custom Data Sample Training**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip install darts

In [2]:
!pip install pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 74.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

# ── 1. Load Data ──────────────────────────────────────────────────────────────
df = pd.read_csv('/content/drive/MyDrive/all_hourly_output.csv')
df['hour_start'] = pd.to_datetime(df['hour_start'])
df['in_session'] = df['in_session'].fillna(0)
df['fce_score']  = df['fce_score'].fillna(0.0)
df = df.sort_values(['room', 'hour_start'])

# ── 2. Add Lags and Targets PER ROOM (critical fix) ───────────────────────────
def add_lags_and_targets(grp):
    for lag in range(1, 49):
        grp[f'occ_lag_{lag}'] = grp['occupancy_now'].shift(lag)
    for i in range(1, 25):
        grp[f'target_h{i}'] = grp['occupancy_now'].shift(-i)
    return grp

df = df.groupby('room', group_keys=False).apply(add_lags_and_targets)
df = df.set_index('hour_start')

# ── 3. Add room as a numeric feature ─────────────────────────────────────────
df['room_code'] = pd.factorize(df['room'])[0]   # e.g. 125b→0, tung→1

# ── 4. Clean ──────────────────────────────────────────────────────────────────
lag_cols    = [f'occ_lag_{i}' for i in range(1, 49)]
target_cols = [f'target_h{i}' for i in range(1, 25)]
features    = ['room_code', 'in_session', 'fce_score', 'capacity'] + lag_cols + ['occupancy_now']

df_clean = df.dropna(subset=features + target_cols)

X = df_clean[features]
y = df_clean[target_cols]

# ── 5. Deduplicated daytime anchors ───────────────────────────────────────────
# Use unique timestamps only — duplicates came from multiple rooms sharing same hour
unique_times = sorted(set(df_clean.index))
test_anchors = unique_times[-5:]   # last 5 unique hours
print("Test anchors:", test_anchors)

# ── 6. Fair Rolling Evaluation ────────────────────────────────────────────────
ridge_maes, xgb_maes = [], []

for anchor in test_anchors:
    X_train = X[X.index < anchor]
    y_train = y[y.index < anchor]

    # Test on ALL rooms at this timestamp (naturally handles multi-room)
    X_test = X[X.index == anchor]
    y_test = y[y.index == anchor]

    if X_train.empty or X_test.empty:
        print(f"  Skipping {anchor}: insufficient data")
        continue

    ridge = Ridge().fit(X_train, y_train)
    xgb   = XGBRegressor(random_state=42, n_jobs=-1).fit(X_train, y_train)

    r_preds = np.clip(ridge.predict(X_test), 0, None).round()
    x_preds = np.clip(xgb.predict(X_test),  0, None).round()
    actuals = y_test.values

    ridge_maes.append(mean_absolute_error(actuals, r_preds))
    xgb_maes.append(mean_absolute_error(actuals, x_preds))
    print(f"  Anchor {anchor}: Ridge={ridge_maes[-1]:.2f}  XGB={xgb_maes[-1]:.2f}")

print(f"\nFair Ridge MAE:   {np.mean(ridge_maes):.2f} people")
print(f"Fair XGBoost MAE: {np.mean(xgb_maes):.2f} people")

# ── 7. Train final models on all data and save ────────────────────────────────
import joblib

final_ridge = Ridge().fit(X, y)
final_xgb   = XGBRegressor(random_state=42, n_jobs=-1).fit(X, y)

joblib.dump(final_ridge, 'occupancy_ridge.pkl')
joblib.dump(final_xgb,   'occupancy_xgb.pkl')
print("\nModels saved: occupancy_ridge.pkl, occupancy_xgb.pkl")
# Add this at the end of your Ridge/XGB training cell
room_codes = dict(zip(df['room'], pd.Categorical(df['room']).codes))
joblib.dump(room_codes, 'room_codes.pkl')

# ── How to reload and predict later ──────────────────────────────────────────
# ridge = joblib.load('occupancy_ridge.pkl')
# xgb   = joblib.load('occupancy_xgb.pkl')
#
# Build a single row with the same features from your DB:
# X_new = pd.DataFrame([{
#     'room_code': 0,          # 0=125b, 1=tung, etc.
#     'in_session': 1,
#     'fce_score': 3.2,
#     'capacity': 25,
#     'occupancy_now': 12,
#     'occ_lag_1': 10, 'occ_lag_2': 8, ...  # last 48h of occupancy
# }])
# preds = np.clip(xgb.predict(X_new)[0], 0, None).round()
# # preds is a 24-element array: next 24 hours per room

/tmp/ipykernel_1404/1447915595.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('room', group_keys=False).apply(add_lags_and_targets)


Test anchors: [Timestamp('2026-04-21 03:00:00'), Timestamp('2026-04-21 04:00:00'), Timestamp('2026-04-21 05:00:00'), Timestamp('2026-04-21 06:00:00'), Timestamp('2026-04-21 07:00:00')]
  Anchor 2026-04-21 03:00:00: Ridge=1.85  XGB=1.62
  Anchor 2026-04-21 04:00:00: Ridge=1.80  XGB=1.64
  Anchor 2026-04-21 05:00:00: Ridge=1.80  XGB=1.67
  Anchor 2026-04-21 06:00:00: Ridge=1.85  XGB=1.65
  Anchor 2026-04-21 07:00:00: Ridge=1.78  XGB=1.74

Fair Ridge MAE:   1.82 people
Fair XGBoost MAE: 1.66 people

Models saved: occupancy_ridge.pkl, occupancy_xgb.pkl


['room_codes.pkl']

## **for 140 data**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

# 1. Load Data
df = pd.read_csv('/content/drive/MyDrive/all_hourly_output.csv')
df['hour_start'] = pd.to_datetime(df['hour_start'])
df = df.sort_values('hour_start').set_index('hour_start')
df['in_session'] = df['in_session'].fillna(0)
df['fce_score'] = df['fce_score'].fillna(0.0)

# 2. Add History and Targets
for lag in range(1, 49):
    df[f'occ_lag_{lag}'] = df['occupancy_now'].shift(lag)
for i in range(1, 25):
    df[f'target_h{i}'] = df['occupancy_now'].shift(-i)

# 3. Clean and Extract Anchors
lag_cols = [f'occ_lag_{i}' for i in range(1, 49)]
target_cols = [f'target_h{i}' for i in range(1, 25)]
features = ['in_session', 'fce_score', 'capacity'] + lag_cols + ['occupancy_now']

df_clean = df.dropna(subset=features + target_cols)

# THE FIX: Explicit Timestamps (The exact start times of the final 5 exams)
test_windows = 20
test_anchors = df_clean.index[-test_windows:]
print("Explicit Test Anchors:", test_anchors.values)

X = df_clean[features]
y = df_clean[target_cols]

# 4. Fair Rolling Evaluation via Timestamps
ridge_maes, xgb_maes = [], []

for anchor in test_anchors:
    # Strict Time Slicing
    X_train = X[X.index < anchor]
    y_train = y[y.index < anchor]
    X_test = X.loc[[anchor]]
    y_test = y.loc[[anchor]]

    ridge = Ridge().fit(X_train, y_train)
    xgb = XGBRegressor(random_state=42, n_jobs=-1).fit(X_train, y_train)

    r_preds = np.clip(ridge.predict(X_test)[0], 0, None).round()
    x_preds = np.clip(xgb.predict(X_test)[0], 0, None).round()
    actuals = y_test.values[0]

    ridge_maes.append(mean_absolute_error(actuals, r_preds))
    xgb_maes.append(mean_absolute_error(actuals, x_preds))

print(f"Fair Ridge MAE: {np.mean(ridge_maes):.2f} people")
print(f"Fair XGBoost MAE: {np.mean(xgb_maes):.2f} people")

Explicit Test Anchors: ['2026-04-21T21:00:00.000000000' '2026-04-21T21:00:00.000000000'
 '2026-04-21T21:00:00.000000000' '2026-04-21T21:00:00.000000000'
 '2026-04-21T22:00:00.000000000' '2026-04-21T22:00:00.000000000'
 '2026-04-21T22:00:00.000000000' '2026-04-21T22:00:00.000000000'
 '2026-04-21T23:00:00.000000000' '2026-04-21T23:00:00.000000000'
 '2026-04-21T23:00:00.000000000' '2026-04-21T23:00:00.000000000'
 '2026-04-22T00:00:00.000000000' '2026-04-22T00:00:00.000000000'
 '2026-04-22T00:00:00.000000000' '2026-04-22T00:00:00.000000000'
 '2026-04-22T01:00:00.000000000' '2026-04-22T01:00:00.000000000'
 '2026-04-22T01:00:00.000000000' '2026-04-22T01:00:00.000000000']
Fair Ridge MAE: 2.03 people
Fair XGBoost MAE: 1.78 people


## **sample**

In [ ]:
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

# 1. Load data
df = pd.read_csv('/content/drive/MyDrive/tung_hourly_output.csv')
df['hour_start'] = pd.to_datetime(df['hour_start'])
df = df.sort_values('hour_start').set_index('hour_start')

# 2. Clean missing values (The Empty Classrooms)
df['in_session'] = df['in_session'].fillna(0)
df['course_id'] = df['course_id'].fillna('None')
df['fce_score'] = df['fce_score'].fillna(0.0)

# 3. Create basic temporal features
df['hour'] = df.index.hour
df['day_of_week'] = df.index.dayofweek

# 4. Select Features (X)
# We will drop course_id for the strict baseline and rely on 'in_session' and 'fce_score'
features = ['occupancy_now', 'temperature', 'condition', 'in_session', 'fce_score', 'hour', 'day_of_week', 'capacity']
X = df[features]

# 5. Select 24-hour targets (Y)
target_cols = [f'occupancy_h{i}' for i in range(1, 25)]
y = df[target_cols]

# 6. Drop the trailing rows that don't have future targets yet
clean_idx = y.dropna().index
X_clean = X.loc[clean_idx]
y_clean = y.loc[clean_idx]

# 7. Split: Train on earlier data, test on the last 24 available hours
split_idx = -24
X_train, X_test = X_clean.iloc[:split_idx], X_clean.iloc[split_idx:]
y_train, y_test = y_clean.iloc[:split_idx], y_clean.iloc[split_idx:]

print(f"Training on {len(X_train)} hours, Testing on {len(X_test)} hours.")

# 8. Train the Models
print("Training Baseline Ridge...")
ridge = Ridge()
ridge.fit(X_train, y_train)

print("Training XGBoost Multi-Output...")
xgb = XGBRegressor(random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)

# 9. Predict and Evaluate
ridge_preds = ridge.predict(X_test)
xgb_preds = xgb.predict(X_test)

mae_ridge = mean_absolute_error(y_test, ridge_preds)
mae_xgb = mean_absolute_error(y_test, xgb_preds)

print(f"Ridge MAE (Across all 24 hours): {mae_ridge:.2f} people")
print(f"XGBoost MAE (Across all 24 hours): {mae_xgb:.2f} people")

Training on 92 hours, Testing on 24 hours.
Training Baseline Ridge...
Training XGBoost Multi-Output...
Ridge MAE (Across all 24 hours): 4.38 people
XGBoost MAE (Across all 24 hours): 1.56 people


## **TFT**

In [ ]:
import numpy as np
import pandas as pd
from darts import TimeSeries
from darts.models import TFTModel
from sklearn.metrics import mean_absolute_error

# ── 1. Clean the dataframe ────────────────────────────────────────────────────
df_clean = df.copy()
df_clean['hour_start'] = pd.to_datetime(df_clean['hour_start'])
df_clean = df_clean.drop_duplicates(subset=['room', 'hour_start']).sort_values(['room', 'hour_start'])

# ── 2. Fill NaNs ──────────────────────────────────────────────────────────────
df_clean['occupancy_now'] = df_clean['occupancy_now'].fillna(0)
df_clean[['in_session', 'fce_score', 'capacity']] = (
    df_clean[['in_session', 'fce_score', 'capacity']].ffill().bfill()
)

# ── 3. Extend covariates to cover full prediction windows ─────────────────────
pad_until = pd.Timestamp('2026-04-23 01:00:00')
extended_rows = []
for room_id, grp in df_clean.groupby('room'):
    last_row = grp.sort_values('hour_start').iloc[-1]
    future_times = pd.date_range(
        start=last_row['hour_start'] + pd.Timedelta(hours=1),
        end=pad_until, freq='h'
    )
    if len(future_times) == 0:
        continue
    extended_rows.append(pd.DataFrame({
        'room':          room_id,
        'hour_start':    future_times,
        'occupancy_now': 0,
        'in_session':    0,
        'fce_score':     last_row['fce_score'],
        'capacity':      last_row['capacity'],
    }))

if extended_rows:
    df_clean = pd.concat([df_clean, *extended_rows], ignore_index=True)
    df_clean = df_clean.sort_values(['room', 'hour_start'])

# ── 4. Build per-room TimeSeries lists ───────────────────────────────────────
target_series = TimeSeries.from_group_dataframe(
    df_clean, group_cols='room', time_col='hour_start',
    value_cols='occupancy_now', fill_missing_dates=True, freq='h',
    static_cols=[]
)
covariates = TimeSeries.from_group_dataframe(
    df_clean, group_cols='room', time_col='hour_start',
    value_cols=['in_session', 'fce_score', 'capacity'],
    fill_missing_dates=True, freq='h',
    static_cols=[]
)

# ── 5. Use DAYTIME anchors (where occupancy actually varies) ──────────────────
unique_anchors = [
    pd.Timestamp('2026-04-21 08:00:00'),
    pd.Timestamp('2026-04-21 09:00:00'),
    pd.Timestamp('2026-04-21 10:00:00'),
    pd.Timestamp('2026-04-21 11:00:00'),
    pd.Timestamp('2026-04-21 12:00:00'),
]

# ── 6. Diagnostic: confirm what hours we're actually testing ──────────────────
print("Test window coverage:")
data_end = pd.Timestamp('2026-04-22 07:00:00')
for anchor in unique_anchors:
    actual_end = min(anchor + pd.Timedelta(hours=23), data_end)
    hours = int((actual_end - anchor).total_seconds() / 3600)
    print(f"  {anchor}  →  {hours}h of actuals (ends {actual_end})")

# ... (keep everything above the eval loop identical) ...

# ── 7. Evaluation loop (unchanged) ───────────────────────────────────────────
tft_maes = []
print("\nRunning Time-Aligned TFT Evaluation...")
for anchor in unique_anchors:
    train_end = anchor - pd.Timedelta(hours=1)
    test_end  = anchor + pd.Timedelta(hours=23)

    train_target, train_covs, val_target = [], [], []
    for ts, cov in zip(target_series, covariates):
        if ts.start_time() >= anchor:
            continue
        if ts.end_time() < anchor:
            continue
        train_target.append(ts.slice(ts.start_time(), train_end))
        train_covs.append(cov)
        actual_test_end = min(test_end, data_end)
        val_target.append(ts.slice(anchor, actual_test_end))

    if not train_target:
        print(f"  Skipping {anchor}: no rooms with sufficient data")
        continue

    tft = TFTModel(
        input_chunk_length=48, output_chunk_length=24, hidden_size=16,
        lstm_layers=1, num_attention_heads=4, batch_size=8, n_epochs=15,
        random_state=42, use_static_covariates=False
    )
    tft.fit(train_target, future_covariates=train_covs, verbose=False)
    preds = tft.predict(n=24, series=train_target, future_covariates=train_covs)

    clean_preds = np.concatenate([
        np.clip(p.values().flatten(), 0, None).round()[:len(v.values().flatten())]
        for p, v in zip(preds, val_target)
    ])
    actuals = np.concatenate([v.values().flatten() for v in val_target])

    mae = mean_absolute_error(actuals, clean_preds)
    tft_maes.append(mae)
    print(f"  Anchor {anchor}: MAE = {mae:.2f}  (over {len(actuals)} hours)")

print(f"\nFair TFT MAE: {np.mean(tft_maes):.2f} people")

# ── 8. Train FINAL model on ALL data and save ─────────────────────────────────
print("\nTraining final model on full dataset...")

# Use the real data only (not the padded future rows) for training targets
real_data_end = pd.Timestamp('2026-04-22 07:00:00')
final_train_target = [ts.slice(ts.start_time(), real_data_end) for ts in target_series]
final_train_covs   = covariates  # covariates already extend to pad_until

final_tft = TFTModel(
    input_chunk_length=48, output_chunk_length=24, hidden_size=16,
    lstm_layers=1, num_attention_heads=4, batch_size=8, n_epochs=15,
    random_state=42, use_static_covariates=False
)
final_tft.fit(final_train_target, future_covariates=final_train_covs, verbose=False)

# Save model — creates occupancy_tft.pt and occupancy_tft.pt.ckpt
final_tft.save("occupancy_tft.pt")
print("Model saved to occupancy_tft.pt")

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Test window coverage:
  2026-04-21 08:00:00  →  23h of actuals (ends 2026-04-22 07:00:00)
  2026-04-21 09:00:00  →  22h of actuals (ends 2026-04-22 07:00:00)
  2026-04-21 10:00:00  →  21h of actuals (ends 2026-04-22 07:00:00)
  2026-04-21 11:00:00  →  20h of actuals (ends 2026-04-22 07:00:00)
  2026-04-21 12:00:00  →  19h of actuals (ends 2026-04-22 07:00:00)

Running Time-Aligned TFT Evaluation...


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=15` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Output()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


  Anchor 2026-04-21 12:00:00: MAE = 1.30  (over 80 hours)

Fair TFT MAE: 1.40 people

Training final model on full dataset...


INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=15` reached.


Model saved to occupancy_tft.pt


In [ ]:
# Run this in your Colab notebook AFTER training
# It saves just the weights separately, avoiding the numpy BitGenerator issue

import torch

# Save the full model config + weights in a portable way
final_tft.save("occupancy_tft.pt")

# Also save weights-only checkpoint for cross-environment loading
torch.save(
    final_tft.model.state_dict(),
    "occupancy_tft_weights.pt"
)
print("Saved both files — download both occupancy_tft.pt and occupancy_tft.pt.ckpt")

Saved both files — download both occupancy_tft.pt and occupancy_tft.pt.ckpt


## **TimeLLM**

In [7]:
%%capture
!pip install neuralforecast

In [8]:
from neuralforecast.utils import AirPassengersDF
#testing
Y_df = AirPassengersDF
Y_df.head()

,unique_id,ds,y
0,1.0,1949-01-31,112.0
1,1.0,1949-02-28,118.0
2,1.0,1949-03-31,132.0
3,1.0,1949-04-30,129.0
4,1.0,1949-05-31,121.0


In [9]:
"""
sweep_all_models_fixed.py
────────────────────────────────────────────────────────────────────────────────
Full sweep: Resolution × Lag → Ridge, XGB, TFT, TimeLLM

FIXES vs original sweep_all_models.py:
  FIX 1 — TFT 45min all-NaN: dataset_45min has UTC tz-aware timestamps.
           Darts TimeSeries.from_dataframe fails on tz-aware data.
           Fix: strip timezone in load_dataset before any model sees the data.

  FIX 2 — in_session construction (45min / 30min datasets):
           Original blindly used class_active→in_session but never checked
           if in_session already existed with real values. Added priority chain:
           in_session (if populated) > class_active > course_id > zeros.

  FIX 3 — fce_score fill: original filled ALL nulls with 0.0 including class rows.
           Now: class rows with null fce_score → per-room median of known scores.
           Non-class rows → 0.0.

  FIX 4 — TFT add_encoders: original used "hour" / "weekday" as encoder keys.
           Correct Darts keys are "hour" and "dayofweek".

Usage:
  python sweep_all_models_fixed.py                        # all models
  python sweep_all_models_fixed.py --models ridge xgb     # sklearn only (fast)
  python sweep_all_models_fixed.py --models tft timellm   # deep only
  python sweep_all_models_fixed.py --metric rmse          # rank by RMSE

Output:
  sweep_results_fixed.csv   — full results table
  best_config_fixed.json    — best config per model
"""

import argparse
import json
import gc
import warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

DATASETS = {
    60: "training_data.csv",
    45: "dataset_45min.csv",
    30: "dataset_30min.csv",
}
LAG_HOURS = [0, 12, 24, 48, 72, 96]
EVAL_FRAC = 0.2


# ═══════════════════════════════════════════════════════════════════════════════
# Data loading
# ═══════════════════════════════════════════════════════════════════════════════

def load_dataset(resolution_min: int) -> pd.DataFrame:
    path = DATASETS[resolution_min]
    df   = pd.read_csv(path)

    time_col = "hour_start" if resolution_min == 60 else "time_bucket"
    df[time_col] = pd.to_datetime(df[time_col])

    # FIX 1: Strip timezone — dataset_45min.csv has UTC tz-aware timestamps.
    # Darts TimeSeries.from_dataframe cannot handle tz-aware DatetimeIndex
    # and silently produces all-NaN results. Must strip tz before any use.
    if df[time_col].dt.tz is not None:
        df[time_col] = df[time_col].dt.tz_localize(None)

    df = df.rename(columns={time_col: "time_bucket"})
    df = df.sort_values(["room", "time_bucket"]).reset_index(drop=True)
    df = df.drop_duplicates(subset=["room", "time_bucket"], keep="last")

    # Targets — normalise to occupancy_step_1..N
    if resolution_min == 60:
        for i in range(1, 24):
            df[f"occupancy_step_{i}"] = df[f"occupancy_h{i}"]
        df["occupancy_step_24"] = df.groupby("room")["occupancy_now"].shift(-24)

    # FIX 2: in_session — priority chain with proper null check
    if "in_session" in df.columns and df["in_session"].notna().sum() > 0:
        df["in_session"] = df["in_session"].fillna(0).astype(int)
    elif "class_active" in df.columns and df["class_active"].notna().sum() > 0:
        df["in_session"] = df["class_active"].fillna(0).astype(int)
    elif "course_id" in df.columns:
        df["in_session"] = df["course_id"].notna().astype(int)
    else:
        df["in_session"] = 0

    n_active = df["in_session"].sum()
    n_total  = len(df)
    print(f"    in_session: {n_active}/{n_total} active ({n_active/n_total:.1%})")

    # Time features
    if "day_of_week_num" in df.columns:
        df["day_of_week"] = df["day_of_week_num"]
    else:
        df["day_of_week"] = df["time_bucket"].dt.dayofweek
    df["hour_of_day"] = df["time_bucket"].dt.hour
    df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)
    if "bucket_in_hour" not in df.columns:
        df["bucket_in_hour"] = 0

    # FIX 3: fce_score smart fill — class rows get per-room median, not 0
    for col in ["fce_score", "extra_hours"]:
        if col not in df.columns:
            df[col] = 0.0
            continue
        room_medians  = df.loc[df["in_session"] == 1].groupby("room")[col].median()
        global_median = df.loc[df["in_session"] == 1, col].median()
        if pd.isna(global_median):
            global_median = 0.0

        def fill_col(grp, col=col, room_medians=room_medians, gm=global_median):
            grp = grp.copy()
            room = grp["room"].iloc[0]
            med  = room_medians.get(room, gm)
            if pd.isna(med):
                med = gm
            grp.loc[(grp["in_session"] == 1) & grp[col].isna(), col] = med
            grp.loc[grp["in_session"] == 0, col] = grp.loc[grp["in_session"] == 0, col].fillna(0.0)
            return grp

        df = df.groupby("room", group_keys=False).apply(fill_col)
        df[col] = df[col].fillna(0.0)

    df["capacity"] = df["capacity"].fillna(
        df.groupby("room")["capacity"].transform("median"))

    for w in ["temperature", "precipitation", "snowfall", "windspeed"]:
        if w not in df.columns:
            df[w] = 0.0
        else:
            df[w] = (df.groupby("room")[w]
                     .transform(lambda s: s.ffill().bfill())
                     .fillna(0.0))

    drop_cols = ["holiday_date", "holiday_in_session", "holiday_description",
                 "source_last_update_utc", "source_last_update_local",
                 "weather_timestamp", "condition", "room_name",
                 "class_name", "day_of_week_num", "course_id",
                 "start_time", "end_time", "  "]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    return df


def build_features(df: pd.DataFrame, resolution_min: int, lag_hours: int) -> tuple:
    steps_per_hour = 60 / resolution_min
    lag_steps      = int(lag_hours * steps_per_hour)
    n_targets      = int(60 * 24 / resolution_min)
    target_cols    = [f"occupancy_step_{i}" for i in range(1, n_targets + 1)]

    df = df.copy()
    df["room_code"] = pd.Categorical(df["room"]).codes

    lag_cols = []
    if lag_steps > 0:
        def add_lags(grp):
            grp = grp.copy()
            for lag in range(1, lag_steps + 1):
                grp[f"occ_lag_{lag}"] = grp["occupancy_now"].shift(lag)
            return grp
        df = df.groupby("room", group_keys=False).apply(add_lags)
        lag_cols = [f"occ_lag_{i}" for i in range(1, lag_steps + 1)]

    def add_future_covs(grp):
        grp = grp.copy()
        for h in range(1, n_targets + 1):
            grp[f"in_session_f{h}"] = grp["in_session"].shift(-h)
            grp[f"fce_f{h}"]        = grp["fce_score"].shift(-h)
            grp[f"hour_f{h}"]       = grp["hour_of_day"].shift(-h)
        return grp
    df = df.groupby("room", group_keys=False).apply(add_future_covs)

    future_cols = (
        [f"in_session_f{h}" for h in range(1, n_targets + 1)] +
        [f"fce_f{h}"        for h in range(1, n_targets + 1)] +
        [f"hour_f{h}"       for h in range(1, n_targets + 1)]
    )
    feature_cols = (
        ["room_code", "hour_of_day", "bucket_in_hour",
         "day_of_week", "is_weekend", "in_session", "fce_score", "capacity"]
        + lag_cols + future_cols
    )
    return df, feature_cols, target_cols, lag_steps, n_targets


# ═══════════════════════════════════════════════════════════════════════════════
# Ridge + XGB
# ═══════════════════════════════════════════════════════════════════════════════

def eval_sklearn(df, feature_cols, target_cols, model_name):
    all_maes, all_rmses = [], []
    for room, grp in df.groupby("room"):
        grp = grp.dropna(subset=feature_cols + target_cols).reset_index(drop=True)
        if len(grp) < 20:
            continue
        split   = int(len(grp) * (1 - EVAL_FRAC))
        X_tr, y_tr = grp.iloc[:split][feature_cols].values, grp.iloc[:split][target_cols].values
        X_te, y_te = grp.iloc[split:][feature_cols].values, grp.iloc[split:][target_cols].values
        if model_name == "ridge":
            m = Ridge(alpha=1.0)
        else:
            m = MultiOutputRegressor(
                XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8,
                             random_state=42, n_jobs=1), n_jobs=1)
        m.fit(X_tr, y_tr)
        preds = np.clip(m.predict(X_te), 0, None)
        all_maes.append(mean_absolute_error(y_te, preds))
        all_rmses.append(np.sqrt(mean_squared_error(y_te, preds)))
    return float(np.mean(all_maes)), float(np.mean(all_rmses))


def train_full_sklearn(df, feature_cols, target_cols, model_name):
    clean = df.dropna(subset=feature_cols + target_cols)
    X, y  = clean[feature_cols].values, clean[target_cols].values
    if model_name == "ridge":
        m = Ridge(alpha=1.0)
    else:
        m = MultiOutputRegressor(
            XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         random_state=42, n_jobs=1), n_jobs=1)
    m.fit(X, y)
    return m


# ═══════════════════════════════════════════════════════════════════════════════
# TFT
# ═══════════════════════════════════════════════════════════════════════════════

def eval_tft(df_raw, resolution_min, lag_steps, n_targets):
    from darts import TimeSeries
    from darts.models import TFTModel
    from darts.dataprocessing.transformers import Scaler

    freq      = f"{resolution_min}min" if resolution_min < 60 else "h"
    timedelta = pd.Timedelta(minutes=resolution_min)

    df = df_raw.copy()
    # FIX 1 (redundant safety): ensure no tz-aware timestamps reach Darts
    if df["time_bucket"].dt.tz is not None:
        df["time_bucket"] = df["time_bucket"].dt.tz_localize(None)

    df["occupancy_now"] = df["occupancy_now"].fillna(0)
    rooms_sorted = sorted(df["room"].unique())

    data_end  = df["time_bucket"].max()
    pad_until = data_end + pd.Timedelta(hours=24)
    ext = []
    for room, grp in df.groupby("room"):
        last  = grp.sort_values("time_bucket").iloc[-1]
        times = pd.date_range(last["time_bucket"] + timedelta, pad_until, freq=freq)
        for t in times:
            ext.append({"room": room, "time_bucket": t, "occupancy_now": np.nan,
                        "in_session": 0, "fce_score": last["fce_score"],
                        "capacity": last["capacity"],
                        "hour_of_day": t.hour, "day_of_week": t.dayofweek})
    df_ext  = pd.concat([df, pd.DataFrame(ext)], ignore_index=True).sort_values(["room", "time_bucket"])
    df_hist = df_ext[df_ext["time_bucket"] <= data_end].copy()
    df_covs = df_ext.copy()
    df_covs["occupancy_now"] = df_covs["occupancy_now"].fillna(0)

    def build_ts(data, vcols, target=False):
        out = []
        for room in rooms_sorted:
            grp = data[data["room"] == room].sort_values("time_bucket")
            if target:
                grp = grp.dropna(subset=["occupancy_now"])
            out.append(TimeSeries.from_dataframe(
                grp, time_col="time_bucket", value_cols=vcols,
                fill_missing_dates=True, freq=freq))
        return out

    targets    = build_ts(df_hist, "occupancy_now", target=True)
    covs       = build_ts(df_covs, ["in_session", "fce_score", "hour_of_day", "day_of_week"])
    scaler     = Scaler()
    targets_sc = scaler.fit_transform(targets)

    split_time = pd.Timestamp(df_hist["time_bucket"].quantile(0.8))
    train_sc   = [ts.drop_after(split_time) for ts in targets_sc]
    val_sc     = [ts.drop_before(split_time) for ts in targets_sc]

    input_len = max(lag_steps, 2)

    # FIX 4: correct Darts encoder keys are "hour" and "dayofweek" (not "weekday")
    tft = TFTModel(
        input_chunk_length=input_len,
        output_chunk_length=n_targets,
        hidden_size=64, lstm_layers=2, num_attention_heads=4,
        dropout=0.1, batch_size=16, n_epochs=30,
        random_state=42, use_static_covariates=False,
        add_encoders={
            "cyclic":   {"future": ["hour", "dayofweek"]},
            "position": {"past":   ["relative"], "future": ["relative"]},
        }
    )
    tft.fit(train_sc, future_covariates=covs, verbose=False)
    preds_sc  = tft.predict(n=n_targets, series=train_sc, future_covariates=covs)
    preds_inv = scaler.inverse_transform(preds_sc)
    vals_inv  = scaler.inverse_transform(val_sc)

    all_preds, all_actuals = [], []
    for p, v in zip(preds_inv, vals_inv):
        pv = np.clip(np.nan_to_num(p.values().flatten()), 0, None)
        av = np.nan_to_num(v.values().flatten())
        mn = min(len(pv), len(av))
        all_preds.extend(pv[:mn])
        all_actuals.extend(av[:mn])

    mae  = mean_absolute_error(all_actuals, all_preds)
    rmse = np.sqrt(mean_squared_error(all_actuals, all_preds))

    import torch
    torch.cuda.empty_cache()
    gc.collect()
    return mae, rmse, tft, scaler, rooms_sorted


def train_full_tft(df_raw, resolution_min, lag_steps, n_targets):
    from darts import TimeSeries
    from darts.models import TFTModel
    from darts.dataprocessing.transformers import Scaler

    freq      = f"{resolution_min}min" if resolution_min < 60 else "h"
    timedelta = pd.Timedelta(minutes=resolution_min)

    df = df_raw.copy()
    if df["time_bucket"].dt.tz is not None:
        df["time_bucket"] = df["time_bucket"].dt.tz_localize(None)

    df["occupancy_now"] = df["occupancy_now"].fillna(0)
    rooms_sorted = sorted(df["room"].unique())

    data_end  = df["time_bucket"].max()
    pad_until = data_end + pd.Timedelta(hours=24)
    ext = []
    for room, grp in df.groupby("room"):
        last  = grp.sort_values("time_bucket").iloc[-1]
        times = pd.date_range(last["time_bucket"] + timedelta, pad_until, freq=freq)
        for t in times:
            ext.append({"room": room, "time_bucket": t, "occupancy_now": np.nan,
                        "in_session": 0, "fce_score": last["fce_score"],
                        "capacity": last["capacity"],
                        "hour_of_day": t.hour, "day_of_week": t.dayofweek})
    df_ext  = pd.concat([df, pd.DataFrame(ext)], ignore_index=True).sort_values(["room", "time_bucket"])
    df_hist = df_ext[df_ext["time_bucket"] <= data_end].copy()
    df_covs = df_ext.copy()
    df_covs["occupancy_now"] = df_covs["occupancy_now"].fillna(0)

    def build_ts(data, vcols, target=False):
        out = []
        for room in rooms_sorted:
            grp = data[data["room"] == room].sort_values("time_bucket")
            if target:
                grp = grp.dropna(subset=["occupancy_now"])
            out.append(TimeSeries.from_dataframe(
                grp, time_col="time_bucket", value_cols=vcols,
                fill_missing_dates=True, freq=freq))
        return out

    targets    = build_ts(df_hist, "occupancy_now", target=True)
    covs       = build_ts(df_covs, ["in_session", "fce_score", "hour_of_day", "day_of_week"])
    scaler     = Scaler()
    targets_sc = scaler.fit_transform(targets)

    input_len = max(lag_steps, 2)
    tft = TFTModel(
        input_chunk_length=input_len, output_chunk_length=n_targets,
        hidden_size=64, lstm_layers=2, num_attention_heads=4,
        dropout=0.1, batch_size=16, n_epochs=30,
        random_state=42, use_static_covariates=False,
        add_encoders={
            "cyclic":   {"future": ["hour", "dayofweek"]},
            "position": {"past":   ["relative"], "future": ["relative"]},
        }
    )
    tft.fit(targets_sc, future_covariates=covs, verbose=False)
    return tft, scaler, rooms_sorted


# ═══════════════════════════════════════════════════════════════════════════════
# TimeLLM
# ═══════════════════════════════════════════════════════════════════════════════

def eval_timellm(df_raw, resolution_min, lag_steps, n_targets):
    import torch
    from neuralforecast import NeuralForecast
    from neuralforecast.models import TimeLLM

    freq  = f"{resolution_min}min" if resolution_min < 60 else "h"
    nf_df = (df_raw[["room", "time_bucket", "occupancy_now"]]
             .rename(columns={"room": "unique_id", "time_bucket": "ds", "occupancy_now": "y"})
             .drop_duplicates(subset=["unique_id", "ds"], keep="last")
             .sort_values(["unique_id", "ds"]).reset_index(drop=True))

    train_rows, test_rows = [], []
    for uid, grp in nf_df.groupby("unique_id"):
        grp   = grp.sort_values("ds").reset_index(drop=True)
        split = int(len(grp) * (1 - EVAL_FRAC))
        train_rows.append(grp.iloc[:split])
        test_rows.append(grp.iloc[split:])
    train_df = pd.concat(train_rows).reset_index(drop=True)
    test_df  = pd.concat(test_rows).reset_index(drop=True)

    input_size    = max(lag_steps, 2)
    prompt_prefix = (
        f"Hourly occupancy data for a university building at {resolution_min}-min resolution. "
        "Rooms hold 25-75 people. Classes run weekdays 8am-8pm."
    )
    timellm = TimeLLM(
        h=n_targets, input_size=input_size,
        llm="openai-community/gpt2",
        prompt_prefix=prompt_prefix,
        batch_size=4, valid_batch_size=4,
        windows_batch_size=4, max_steps=100,
    )
    nf        = NeuralForecast(models=[timellm], freq=freq)
    nf.fit(df=train_df)
    forecasts = nf.predict(df=train_df)
    merged    = forecasts.merge(test_df.rename(columns={"y": "actual"}),
                                on=["unique_id", "ds"], how="inner")
    if merged.empty:
        return float("inf"), float("inf"), None

    preds   = merged["TimeLLM"].clip(lower=0).round().values
    actuals = merged["actual"].values
    mae     = mean_absolute_error(actuals, preds)
    rmse    = np.sqrt(mean_squared_error(actuals, preds))

    del timellm, nf, forecasts, merged
    gc.collect()
    torch.cuda.empty_cache()
    return mae, rmse, None


def train_full_timellm(df_raw, resolution_min, lag_steps, n_targets):
    from neuralforecast import NeuralForecast
    from neuralforecast.models import TimeLLM

    freq  = f"{resolution_min}min" if resolution_min < 60 else "h"
    nf_df = (df_raw[["room", "time_bucket", "occupancy_now"]]
             .rename(columns={"room": "unique_id", "time_bucket": "ds", "occupancy_now": "y"})
             .drop_duplicates(subset=["unique_id", "ds"], keep="last")
             .sort_values(["unique_id", "ds"]).reset_index(drop=True))

    input_size    = max(lag_steps, 2)
    prompt_prefix = (
        f"University building occupancy at {resolution_min}-min resolution. "
        "Classes weekdays 8am-8pm. Rooms hold 25-75 people."
    )
    timellm = TimeLLM(
        h=n_targets, input_size=input_size,
        llm="openai-community/gpt2",
        prompt_prefix=prompt_prefix,
        batch_size=4, valid_batch_size=4,
        windows_batch_size=4, max_steps=100,
    )
    nf = NeuralForecast(models=[timellm], freq=freq)
    nf.fit(df=nf_df)
    return nf


# ═══════════════════════════════════════════════════════════════════════════════
# Main sweep
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--models", nargs="+",
                        choices=["ridge", "xgb", "tft", "timellm"],
                        default=["ridge", "xgb", "tft", "timellm"])
    parser.add_argument("--metric", choices=["mae", "rmse"], default="mae")
    args, _ = parser.parse_known_args()

    results = []
    best = {m: {"score": float("inf"), "config": None, "model_obj": None}
            for m in args.models}

    print("=" * 65)
    print(f"  SWEEP: All Models × Resolution × Lag  [FIXED]")
    print(f"  Models     : {args.models}")
    print(f"  Resolutions: {list(DATASETS.keys())} min")
    print(f"  Lag windows: {LAG_HOURS} hrs")
    print(f"  Rank by    : {args.metric.upper()}")
    print("=" * 65)

    for res in sorted(DATASETS.keys(), reverse=True):
        print(f"\n{'─'*65}")
        print(f"  Resolution: {res}min  |  {DATASETS[res]}")
        print(f"{'─'*65}")
        df_raw = load_dataset(res)

        for lag_hrs in LAG_HOURS:
            steps_per_hour = 60 / res
            lag_steps  = int(lag_hrs * steps_per_hour)
            n_targets  = int(60 * 24 / res)
            print(f"\n  ▶ lag={lag_hrs}h ({lag_steps} steps)  targets={n_targets} steps")

            row      = {"resolution_min": res, "lag_hrs": lag_hrs,
                        "lag_steps": lag_steps, "n_targets": n_targets}
            df_feat, feature_cols, target_cols, _, _ = build_features(df_raw, res, lag_hrs)

            if "ridge" in args.models:
                mae, rmse = eval_sklearn(df_feat, feature_cols, target_cols, "ridge")
                row["mae_ridge"] = round(mae, 3)
                row["rmse_ridge"] = round(rmse, 3)
                score = mae if args.metric == "mae" else rmse
                print(f"      Ridge    MAE={mae:.3f}  RMSE={rmse:.3f}", end="")
                if score < best["ridge"]["score"]:
                    best["ridge"]["score"]  = score
                    best["ridge"]["config"] = {"res": res, "lag_hrs": lag_hrs,
                                               "lag_steps": lag_steps,
                                               "feature_cols": feature_cols,
                                               "target_cols": target_cols}
                    print("  ★ best", end="")
                print()

            if "xgb" in args.models:
                mae, rmse = eval_sklearn(df_feat, feature_cols, target_cols, "xgb")
                row["mae_xgb"] = round(mae, 3)
                row["rmse_xgb"] = round(rmse, 3)
                score = mae if args.metric == "mae" else rmse
                print(f"      XGB      MAE={mae:.3f}  RMSE={rmse:.3f}", end="")
                if score < best["xgb"]["score"]:
                    best["xgb"]["score"]  = score
                    best["xgb"]["config"] = {"res": res, "lag_hrs": lag_hrs,
                                             "lag_steps": lag_steps,
                                             "feature_cols": feature_cols,
                                             "target_cols": target_cols}
                    print("  ★ best", end="")
                print()

            if "tft" in args.models:
                try:
                    mae, rmse, tft_obj, scaler, rooms = eval_tft(df_raw, res, lag_steps, n_targets)
                    row["mae_tft"]  = round(mae, 3)
                    row["rmse_tft"] = round(rmse, 3)
                    score = mae if args.metric == "mae" else rmse
                    print(f"      TFT      MAE={mae:.3f}  RMSE={rmse:.3f}", end="")
                    if score < best["tft"]["score"]:
                        best["tft"]["score"]     = score
                        best["tft"]["config"]    = {"res": res, "lag_hrs": lag_hrs,
                                                    "lag_steps": lag_steps,
                                                    "n_targets": n_targets}
                        best["tft"]["model_obj"] = (tft_obj, scaler, rooms)
                        print("  ★ best", end="")
                    print()
                except Exception as e:
                    print(f"      TFT      FAILED: {e}")
                    row["mae_tft"] = None
                    row["rmse_tft"] = None

            if "timellm" in args.models:
                try:
                    mae, rmse, _ = eval_timellm(df_raw, res, lag_steps, n_targets)
                    row["mae_timellm"]  = round(mae, 3)
                    row["rmse_timellm"] = round(rmse, 3)
                    score = mae if args.metric == "mae" else rmse
                    print(f"      TimeLLM  MAE={mae:.3f}  RMSE={rmse:.3f}", end="")
                    if score < best["timellm"]["score"]:
                        best["timellm"]["score"]  = score
                        best["timellm"]["config"] = {"res": res, "lag_hrs": lag_hrs,
                                                     "lag_steps": lag_steps,
                                                     "n_targets": n_targets}
                        print("  ★ best", end="")
                    print()
                except Exception as e:
                    print(f"      TimeLLM  FAILED: {e}")
                    row["mae_timellm"] = None
                    row["rmse_timellm"] = None

            results.append(row)

    df_res = pd.DataFrame(results)
    df_res.to_csv("sweep_results_fixed.csv", index=False)

    print(f"\n{'='*65}")
    print("  SWEEP COMPLETE — Results")
    print(f"{'='*65}")
    print(df_res.to_string(index=False))

    # Retrain best of each model on full data
    print(f"\n{'─'*65}")
    print("  Retraining best configs on full data …")
    print(f"{'─'*65}")

    best_configs_out = {}
    for model_name in args.models:
        cfg = best[model_name]["config"]
        if cfg is None:
            continue
        print(f"\n  {model_name.upper()}  →  res={cfg['res']}min  lag={cfg['lag_hrs']}h  "
              f"score={best[model_name]['score']:.3f}")

        df_raw = load_dataset(cfg["res"])
        df_feat, feature_cols, target_cols, lag_steps, n_targets = build_features(
            df_raw, cfg["res"], cfg["lag_hrs"])

        if model_name in ("ridge", "xgb"):
            m = train_full_sklearn(df_feat, feature_cols, target_cols, model_name)
            joblib.dump(m,            f"best_{model_name}.pkl")
            joblib.dump(feature_cols, f"best_{model_name}_feature_cols.pkl")
            joblib.dump(target_cols,  f"best_{model_name}_target_cols.pkl")
            room_codes = dict(zip(df_raw["room"], pd.Categorical(df_raw["room"]).codes))
            joblib.dump(room_codes,   f"best_{model_name}_room_codes.pkl")
            print(f"    Saved: best_{model_name}.pkl")

        elif model_name == "tft":
            tft, scaler, rooms = train_full_tft(df_raw, cfg["res"],
                                                cfg["lag_steps"], cfg["n_targets"])
            tft.save("best_tft.pt")
            joblib.dump(scaler, "best_tft_scaler.pkl")
            joblib.dump(rooms,  "best_tft_room_order.pkl")
            print("    Saved: best_tft.pt, best_tft_scaler.pkl, best_tft_room_order.pkl")

        elif model_name == "timellm":
            nf = train_full_timellm(df_raw, cfg["res"],
                                    cfg["lag_steps"], cfg["n_targets"])
            nf.save("best_timellm/", overwrite=True)
            print("    Saved: best_timellm/")

        best_configs_out[model_name] = {
            "resolution_min": cfg["res"],
            "lag_hrs":        cfg["lag_hrs"],
            "lag_steps":      cfg.get("lag_steps", 0),
            "n_targets":      cfg.get("n_targets", 0),
            f"{args.metric}": best[model_name]["score"],
        }

    with open("best_config_fixed.json", "w") as f:
        json.dump(best_configs_out, f, indent=2)

    print(f"\nSaved: sweep_results_fixed.csv, best_config_fixed.json")
    print("\nBest config per model:")
    print(json.dumps(best_configs_out, indent=2))


if __name__ == "__main__":
    main()

  SWEEP: All Models × Resolution × Lag  [FIXED]
  Models     : ['ridge', 'xgb', 'tft', 'timellm']
  Resolutions: [60, 45, 30] min
  Lag windows: [0, 12, 24, 48, 72, 96] hrs
  Rank by    : MAE

─────────────────────────────────────────────────────────────────
  Resolution: 60min  |  training_data.csv
─────────────────────────────────────────────────────────────────
    in_session: 105/704 active (14.9%)

  ▶ lag=0h (0 steps)  targets=24 steps
      Ridge    MAE=6.737  RMSE=8.256  ★ best


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


      XGB      MAE=3.920  RMSE=6.101  ★ best


INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=30` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

...

┏━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                │ MAE                │      0 │ train │     0 │
│ 1 │ padder_train        │ ConstantPad1d      │      0 │ train │     0 │
│ 2 │ scaler              │ TemporalNorm       │      0 │ train │     0 │
│ 3 │ llm                 │ GPT2Model          │  124 M │ eval  │     0 │
│ 4 │ patch_embedding     │ PatchEmbedding     │  1.5 K │ train │     0 │
│ 5 │ mapping_layer       │ Linear             │ 51.5 M │ train │     0 │
│ 6 │ reprogramming_layer │ ReprogrammingLayer │  2.4 M │ train │     0 │
│ 7 │ output_projection   │ FlattenHead        │ 36.9 K │ train │     0 │
│ 8 │ normalize_layers    │ RevIN              │      0 │ train │     0 │
└───┴─────────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 53.9 M                                                                                           
Non-trainable params: 124 M                                                                                        
Total params: 178 M                                                                                                
Total estimated model params size (MB): 713                                                                        
Modules in train mode: 20                                                                                          
Modules in eval mode: 162                                                                                          
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.


    Saved: best_timellm/

Saved: sweep_results_fixed.csv, best_config_fixed.json

Best config per model:
{
  "ridge": {
    "resolution_min": 60,
    "lag_hrs": 72,
    "lag_steps": 72,
    "n_targets": 0,
    "mae": 1.8444713869507272
  },
  "xgb": {
    "resolution_min": 60,
    "lag_hrs": 96,
    "lag_steps": 96,
    "n_targets": 0,
    "mae": 1.7549055687618251
  },
  "tft": {
    "resolution_min": 30,
    "lag_hrs": 24,
    "lag_steps": 48,
    "n_targets": 48,
    "mae": 0.3627046512185141
  },
  "timellm": {
    "resolution_min": 30,
    "lag_hrs": 24,
    "lag_steps": 48,
    "n_targets": 48,
    "mae": 0.359375
  }
}


In [10]:
"""
feature_ablations_fixed.py
────────────────────────────────────────────────────────────────────────────────
Phase 3 — Feature Ablation  [FIXED v2]

BUGS FIXED vs original feature_ablations.py:
─────────────────────────────────────────────
BUG 1 — TFT: wrong baseline (1.7474 instead of ~0.7)
  CAUSE: eval_tft received sklearn's full feature_cols list (253 cols) and tried
  to map it to TFT covariates, accidentally including weather in the baseline.
  FIX: eval_tft now accepts ablated_group (a group NAME) instead of a feature
  list. TFT_COVARIATE_BASE is now exactly the 4 covariates the sweep used
  [in_session, fce_score, hour_of_day, day_of_week], so ablation FULL baseline
  is directly comparable to sweep results.

BUG 2 — TFT: G5/G6 negative delta (removal "helps")
  CAUSE: TFT never uses sklearn lag columns (G5) or future sklearn cols (G6).
  MAE variation was pure training stochasticity, not feature signal.
  FIX: G5 marked N/A (TFT uses past series directly). G6 maps to schedule
  covariates [in_session, fce_score] — closest valid approximation; flagged
  in notes so readers know it is an approximation not an exact removal.

BUG 3 — TFT: 45min all NaN
  CAUSE: dataset_45min.csv has UTC tz-aware timestamps; Darts fails silently.
  FIX: load_dataset strips timezone before any model sees the data.

BUG 4 — TFT: wrong add_encoders key ("weekday" → "dayofweek")

BUG 5 — Summary averages corrupted by broken TFT stochastic rows
  FIX: rows with note != "" excluded from group average.

Design notes (known limitations, not bugs):
  G3_capacity: capacity is static per room, acts like a room-ID proxy.
    G3 importance measures "does room identity help?" — real but interpret
    carefully; it is not the same as a truly varying capacity signal.
  G6_future for TFT: mapped to [in_session, fce_score] as an approximation.
    True future covariates (in_session_f*) do not exist as separate TFT inputs;
    this is the closest equivalent. Label as "approx." in any report.

TFT covariate mapping per ablated group (4-covariate base, matching sweep):
  FULL  → [in_session, fce_score, hour_of_day, day_of_week]
  -G1   → remove hour_of_day, day_of_week  → [in_session, fce_score]
  -G2   → remove in_session, fce_score     → [hour_of_day, day_of_week]
  -G3   → no TFT covariates map to capacity/extra_hours → N/A
  -G4   → no TFT covariates map to weather in base set  → N/A
  -G5   → N/A (TFT uses past series, not lag columns)
  -G6   → remove in_session, fce_score (approx) → [hour_of_day, day_of_week]

Usage:
  python feature_ablations_fixed.py                        # all 4 models
  python feature_ablations_fixed.py --models ridge xgb     # sklearn only
  python feature_ablations_fixed.py --models tft           # TFT only

Output:
  ablation_results_fixed.csv   — full ΔMAE table
  ablation_summary_fixed.csv   — ranked importance (Ridge + XGB + TFT; TimeLLM excluded)
"""

import argparse
import gc
import warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# ── Best configs from sweep ───────────────────────────────────────────────────
BEST_CONFIGS = {
    "ridge":   {"resolution_min": 60, "lag_hrs": 72},
    "xgb":     {"resolution_min": 60, "lag_hrs": 96},
    "tft":     {"resolution_min": 30, "lag_hrs": 48},
    "timellm": {"resolution_min": 30, "lag_hrs": 24},
}

DATASETS = {
    60: "training_data.csv",
    45: "dataset_45min.csv",
    30: "dataset_30min.csv",
}

EVAL_FRAC = 0.2

# ── Feature group definitions (for Ridge / XGB) ───────────────────────────────
FEATURE_GROUPS = {
    "G1_temporal": ["hour_of_day", "day_of_week", "is_weekend", "bucket_in_hour"],
    "G2_schedule": ["in_session", "fce_score"],
    "G3_capacity": ["capacity", "extra_hours"],
    "G4_weather":  ["temperature", "precipitation", "snowfall", "windspeed"],
    "G5_history":  "prefix:occ_lag_",
    "G6_future":   "prefix:in_session_f,fce_f,hour_f",
}

# ── TFT covariate map per ablated group ───────────────────────────────────────
# Lists which covariate columns to REMOVE when that group is ablated.
# Keys must match FEATURE_GROUPS keys exactly.
#
# TFT_COVARIATE_BASE: MUST match the 4 covariates used in sweep_all_models.py
# so the ablation FULL baseline is directly comparable to sweep TFT results.
# Weather and capacity were NOT in the sweep covariate set — adding them here
# would create a stronger baseline than sweep used, making results incomparable.
TFT_COVARIATE_BASE = [
    "in_session", "fce_score", "hour_of_day", "day_of_week",
]

# Maps group → covariate cols to DROP from TFT_COVARIATE_BASE.
# None = group has no equivalent in TFT's covariate set → skip with N/A note.
TFT_GROUP_COVARIATES = {
    "G1_temporal": ["hour_of_day", "day_of_week"],
    "G2_schedule": ["in_session", "fce_score"],
    "G3_capacity": None,   # capacity/extra_hours not in TFT covariate base → N/A
    "G4_weather":  None,   # weather not in TFT covariate base → N/A
    "G5_history":  None,   # TFT uses past series directly, not lag columns → N/A
    "G6_future":   ["in_session", "fce_score"],  # closest approx; flagged in note
}

TFT_GROUP_NOTES = {
    "G3_capacity": "N/A — capacity/extra_hours not in TFT covariate base (sweep-aligned)",
    "G4_weather":  "N/A — weather not in TFT covariate base (sweep-aligned)",
    "G5_history":  "N/A — TFT uses past observed series directly, not explicit lag columns",
    "G6_future":   "approx — mapped to [in_session, fce_score]; exact future cols unavailable in TFT",
}


def resolve_group(group_def, all_feature_cols):
    if isinstance(group_def, list):
        return [c for c in group_def if c in all_feature_cols]
    prefixes = [p.strip() for p in group_def.replace("prefix:", "").split(",")]
    return [c for c in all_feature_cols if any(c.startswith(p) for p in prefixes)]


# ═══════════════════════════════════════════════════════════════════════════════
# Data loading
# ═══════════════════════════════════════════════════════════════════════════════

def load_dataset(resolution_min):
    path = DATASETS[resolution_min]
    df   = pd.read_csv(path)
    time_col = "hour_start" if resolution_min == 60 else "time_bucket"
    df[time_col] = pd.to_datetime(df[time_col])

    # FIX 3: strip timezone — dataset_45min has UTC-aware timestamps
    # Darts TimeSeries.from_dataframe silently produces NaN on tz-aware data
    if df[time_col].dt.tz is not None:
        df[time_col] = df[time_col].dt.tz_localize(None)

    df = df.rename(columns={time_col: "time_bucket"})
    df = df.sort_values(["room", "time_bucket"]).reset_index(drop=True)
    df = df.drop_duplicates(subset=["room", "time_bucket"], keep="last")

    if resolution_min == 60:
        for i in range(1, 24):
            df[f"occupancy_step_{i}"] = df[f"occupancy_h{i}"]
        df["occupancy_step_24"] = df.groupby("room")["occupancy_now"].shift(-24)

    # in_session — priority chain
    if "in_session" in df.columns and df["in_session"].notna().sum() > 0:
        df["in_session"] = df["in_session"].fillna(0).astype(int)
    elif "class_active" in df.columns and df["class_active"].notna().sum() > 0:
        df["in_session"] = df["class_active"].fillna(0).astype(int)
    elif "course_id" in df.columns:
        df["in_session"] = df["course_id"].notna().astype(int)
    else:
        df["in_session"] = 0
        print("  ⚠️  No class/session source found — in_session set to 0")

    n_active = df["in_session"].sum()
    n_total  = len(df)
    print(f"  ✅  in_session: {n_active}/{n_total} rows active ({n_active/n_total:.1%})")

    if "day_of_week_num" in df.columns:
        df["day_of_week"] = df["day_of_week_num"]
    else:
        df["day_of_week"] = df["time_bucket"].dt.dayofweek
    df["hour_of_day"] = df["time_bucket"].dt.hour
    df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)
    if "bucket_in_hour" not in df.columns:
        df["bucket_in_hour"] = 0

    # fce_score / extra_hours smart fill
    for col in ["fce_score", "extra_hours"]:
        if col not in df.columns:
            df[col] = 0.0
            continue
        room_medians  = df.loc[df["in_session"] == 1].groupby("room")[col].median()
        global_median = df.loc[df["in_session"] == 1, col].median()
        if pd.isna(global_median):
            global_median = 0.0

        def fill_col(grp, col=col, room_medians=room_medians, gm=global_median):
            grp = grp.copy()
            room = grp["room"].iloc[0]
            med  = room_medians.get(room, gm)
            if pd.isna(med):
                med = gm
            grp.loc[(grp["in_session"] == 1) & grp[col].isna(), col] = med
            grp.loc[grp["in_session"] == 0, col] = grp.loc[grp["in_session"] == 0, col].fillna(0.0)
            return grp

        df = df.groupby("room", group_keys=False).apply(fill_col)
        df[col] = df[col].fillna(0.0)

    df["capacity"] = df["capacity"].fillna(
        df.groupby("room")["capacity"].transform("median"))

    for w in ["temperature", "precipitation", "snowfall", "windspeed"]:
        if w not in df.columns:
            df[w] = 0.0
        else:
            df[w] = (df.groupby("room")[w]
                     .transform(lambda s: s.ffill().bfill())
                     .fillna(0.0))

    drop_cols = ["holiday_date", "holiday_in_session", "holiday_description",
                 "source_last_update_utc", "source_last_update_local",
                 "weather_timestamp", "condition", "room_name",
                 "class_name", "day_of_week_num", "course_id",
                 "start_time", "end_time", "  "]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    return df


def build_features(df, resolution_min, lag_hours):
    steps_per_hour = 60 / resolution_min
    lag_steps      = int(lag_hours * steps_per_hour)
    n_targets      = int(60 * 24 / resolution_min)
    target_cols    = [f"occupancy_step_{i}" for i in range(1, n_targets + 1)]

    df = df.copy()
    df["room_code"] = pd.Categorical(df["room"]).codes

    lag_cols = []
    if lag_steps > 0:
        def add_lags(grp):
            grp = grp.copy()
            for lag in range(1, lag_steps + 1):
                grp[f"occ_lag_{lag}"] = grp["occupancy_now"].shift(lag)
            return grp
        df = df.groupby("room", group_keys=False).apply(add_lags)
        lag_cols = [f"occ_lag_{i}" for i in range(1, lag_steps + 1)]

    def add_future_covs(grp):
        grp = grp.copy()
        for h in range(1, n_targets + 1):
            grp[f"in_session_f{h}"] = grp["in_session"].shift(-h)
            grp[f"fce_f{h}"]        = grp["fce_score"].shift(-h)
            grp[f"hour_f{h}"]       = grp["hour_of_day"].shift(-h)
        return grp
    df = df.groupby("room", group_keys=False).apply(add_future_covs)

    future_cols = (
        [f"in_session_f{h}" for h in range(1, n_targets + 1)] +
        [f"fce_f{h}"        for h in range(1, n_targets + 1)] +
        [f"hour_f{h}"       for h in range(1, n_targets + 1)]
    )

    base_cols    = ["room_code", "hour_of_day", "bucket_in_hour", "day_of_week",
                    "is_weekend", "in_session", "fce_score", "capacity", "extra_hours",
                    "temperature", "precipitation", "snowfall", "windspeed"]
    feature_cols = base_cols + lag_cols + future_cols
    return df, feature_cols, target_cols, lag_steps, n_targets


# ═══════════════════════════════════════════════════════════════════════════════
# Evaluators
# ═══════════════════════════════════════════════════════════════════════════════

def eval_sklearn(df, feature_cols, target_cols, model_name):
    all_maes = []
    for room, grp in df.groupby("room"):
        grp = grp.dropna(subset=feature_cols + target_cols).reset_index(drop=True)
        if len(grp) < 20:
            continue
        split = int(len(grp) * (1 - EVAL_FRAC))
        X_tr = grp.iloc[:split][feature_cols].values
        y_tr = grp.iloc[:split][target_cols].values
        X_te = grp.iloc[split:][feature_cols].values
        y_te = grp.iloc[split:][target_cols].values
        if model_name == "ridge":
            m = Ridge(alpha=1.0)
        else:
            m = MultiOutputRegressor(
                XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8,
                             random_state=42, n_jobs=1), n_jobs=1)
        m.fit(X_tr, y_tr)
        preds = np.clip(m.predict(X_te), 0, None)
        all_maes.append(mean_absolute_error(y_te, preds))
    return float(np.mean(all_maes))


def eval_tft(df_raw, resolution_min, lag_steps, n_targets, ablated_group=None):
    """
    FIX 1+2: TFT covariate control is now group-name-based, not feature-list-based.

    ablated_group: one of the FEATURE_GROUPS keys (e.g. "G4_weather"), or None for FULL.
    The function looks up TFT_GROUP_COVARIATES[ablated_group] to know which
    covariates to drop. This is the only correct way to ablate TFT —
    passing sklearn column lists was meaningless and caused stochastic artefacts.

    G5_history ablation: TFT's "history" IS its past observed series input.
    There is no covariate column to drop. We report the result honestly and
    flag it as "N/A — TFT uses past series directly, not lag columns".
    """
    import torch
    from darts import TimeSeries
    from darts.models import TFTModel
    from darts.dataprocessing.transformers import Scaler

    freq      = f"{resolution_min}min" if resolution_min < 60 else "h"
    timedelta = pd.Timedelta(minutes=resolution_min)

    df = df_raw.copy()
    # FIX 3: strip tz — dataset_45min has UTC-aware timestamps
    if df["time_bucket"].dt.tz is not None:
        df["time_bucket"] = df["time_bucket"].dt.tz_localize(None)

    df["occupancy_now"] = df["occupancy_now"].fillna(0)
    rooms_sorted = sorted(df["room"].unique())

    # Determine active covariates for this ablation
    # None means the group has no covariate equivalent in TFT → caller handles N/A
    if ablated_group:
        drop_covs = TFT_GROUP_COVARIATES.get(ablated_group)
        if drop_covs is None:
            raise ValueError(f"Group {ablated_group} is N/A for TFT — should be skipped before calling eval_tft")
        active_covs = [c for c in TFT_COVARIATE_BASE if c in df.columns and c not in drop_covs]
    else:
        active_covs = [c for c in TFT_COVARIATE_BASE if c in df.columns]

    if not active_covs:
        active_covs = ["hour_of_day"]   # Darts requires at least one covariate

    # Report what we're actually doing
    if ablated_group:
        note = TFT_GROUP_NOTES.get(ablated_group, "")
        drop_label = drop_covs if drop_covs else "(none)"
        print(f"    TFT dropping: {drop_label}  → active covs: {active_covs}")
        if note:
            print(f"    NOTE: {note}")

    data_end  = df["time_bucket"].max()
    pad_until = data_end + pd.Timedelta(hours=24)
    ext = []
    for room, grp in df.groupby("room"):
        last  = grp.sort_values("time_bucket").iloc[-1]
        times = pd.date_range(last["time_bucket"] + timedelta, pad_until, freq=freq)
        for t in times:
            row = {"room": room, "time_bucket": t, "occupancy_now": np.nan,
                   "in_session": 0, "fce_score": last.get("fce_score", 0.0),
                   "capacity":   last.get("capacity", 0.0),
                   "extra_hours": 0.0, "hour_of_day": t.hour,
                   "day_of_week": t.dayofweek,
                   "temperature": last.get("temperature", 0.0),
                   "precipitation": 0.0, "snowfall": 0.0, "windspeed": 0.0}
            ext.append(row)

    df_ext  = pd.concat([df, pd.DataFrame(ext)], ignore_index=True).sort_values(["room", "time_bucket"])
    df_hist = df_ext[df_ext["time_bucket"] <= data_end].copy()
    df_covs = df_ext.copy()
    df_covs["occupancy_now"] = df_covs["occupancy_now"].fillna(0)

    # Fill any covariate cols that may be missing in padded rows
    for c in active_covs:
        if c not in df_covs.columns:
            df_covs[c] = 0.0
        df_covs[c] = df_covs[c].fillna(0.0)

    def build_ts(data, vcols, target=False):
        out = []
        for room in rooms_sorted:
            grp = data[data["room"] == room].sort_values("time_bucket")
            if target:
                grp = grp.dropna(subset=["occupancy_now"])
            out.append(TimeSeries.from_dataframe(
                grp, time_col="time_bucket", value_cols=vcols,
                fill_missing_dates=True, freq=freq))
        return out

    targets    = build_ts(df_hist, "occupancy_now", target=True)
    covs       = build_ts(df_covs, active_covs)
    scaler     = Scaler()
    targets_sc = scaler.fit_transform(targets)

    split_time = pd.Timestamp(df_hist["time_bucket"].quantile(0.8))
    train_sc   = [ts.drop_after(split_time) for ts in targets_sc]
    val_sc     = [ts.drop_before(split_time) for ts in targets_sc]

    input_len = max(lag_steps, 2)

    # FIX 4: correct Darts encoder key is "dayofweek" not "weekday"
    tft = TFTModel(
        input_chunk_length=input_len,
        output_chunk_length=n_targets,
        hidden_size=64, lstm_layers=2, num_attention_heads=4,
        dropout=0.1, batch_size=16, n_epochs=30,
        random_state=42, use_static_covariates=False,
        add_encoders={
            "cyclic":   {"future": ["hour", "dayofweek"]},
            "position": {"past":   ["relative"], "future": ["relative"]},
        }
    )
    tft.fit(train_sc, future_covariates=covs, verbose=False)
    preds_sc  = tft.predict(n=n_targets, series=train_sc, future_covariates=covs)
    preds_inv = scaler.inverse_transform(preds_sc)
    vals_inv  = scaler.inverse_transform(val_sc)

    all_p, all_a = [], []
    for p, v in zip(preds_inv, vals_inv):
        pv = np.clip(np.nan_to_num(p.values().flatten()), 0, None)
        av = np.nan_to_num(v.values().flatten())
        mn = min(len(pv), len(av))
        all_p.extend(pv[:mn])
        all_a.extend(av[:mn])

    mae = mean_absolute_error(all_a, all_p)
    torch.cuda.empty_cache()
    gc.collect()
    return mae


def eval_timellm(df_raw, resolution_min, lag_steps, n_targets):
    import torch
    from neuralforecast import NeuralForecast
    from neuralforecast.models import TimeLLM

    freq  = f"{resolution_min}min" if resolution_min < 60 else "h"
    nf_df = (df_raw[["room", "time_bucket", "occupancy_now"]]
             .rename(columns={"room": "unique_id", "time_bucket": "ds", "occupancy_now": "y"})
             .drop_duplicates(subset=["unique_id", "ds"], keep="last")
             .sort_values(["unique_id", "ds"]).reset_index(drop=True))

    train_rows, test_rows = [], []
    for uid, grp in nf_df.groupby("unique_id"):
        grp   = grp.sort_values("ds").reset_index(drop=True)
        split = int(len(grp) * (1 - EVAL_FRAC))
        train_rows.append(grp.iloc[:split])
        test_rows.append(grp.iloc[split:])
    train_df = pd.concat(train_rows).reset_index(drop=True)
    test_df  = pd.concat(test_rows).reset_index(drop=True)

    input_size = max(lag_steps, 2)
    timellm = TimeLLM(
        h=n_targets, input_size=input_size,
        llm="openai-community/gpt2",
        prompt_prefix="University building occupancy. Classes weekdays 8am-8pm.",
        batch_size=4, valid_batch_size=4,
        windows_batch_size=4, max_steps=100,
    )
    nf        = NeuralForecast(models=[timellm], freq=freq)
    nf.fit(df=train_df)
    forecasts = nf.predict(df=train_df)
    merged    = forecasts.merge(test_df.rename(columns={"y": "actual"}),
                                on=["unique_id", "ds"], how="inner")
    if merged.empty:
        return float("inf")

    mae = mean_absolute_error(merged["actual"].values,
                              merged["TimeLLM"].clip(lower=0).round().values)
    del timellm, nf, forecasts, merged
    gc.collect()
    torch.cuda.empty_cache()
    return mae


# ═══════════════════════════════════════════════════════════════════════════════
# Main ablation loop
# ═══════════════════════════════════════════════════════════════════════════════

def run_ablation_for_model(model_name):
    cfg     = BEST_CONFIGS[model_name]
    res     = cfg["resolution_min"]
    lag_hrs = cfg["lag_hrs"]

    print(f"\n{'═'*65}")
    print(f"  {model_name.upper()}  |  res={res}min  lag={lag_hrs}h")
    print(f"{'═'*65}")

    df_raw = load_dataset(res)
    df_feat, feature_cols, target_cols, lag_steps, n_targets = build_features(
        df_raw, res, lag_hrs)

    groups = {name: resolve_group(defn, feature_cols)
              for name, defn in FEATURE_GROUPS.items()}

    print(f"\n  Feature groups resolved:")
    for g, cols in groups.items():
        present = [c for c in cols if c in feature_cols]
        print(f"    {g}: {len(present)} cols  {present[:3]}{'...' if len(present) > 3 else ''}")

    results = []

    # ── FULL baseline ─────────────────────────────────────────────────────────
    print(f"\n  [FULL] all features ({len(feature_cols)} cols)")
    if model_name == "ridge":
        baseline_mae = eval_sklearn(df_feat, feature_cols, target_cols, "ridge")
    elif model_name == "xgb":
        baseline_mae = eval_sklearn(df_feat, feature_cols, target_cols, "xgb")
    elif model_name == "tft":
        baseline_mae = eval_tft(df_raw, res, lag_steps, n_targets, ablated_group=None)

    print(f"    MAE = {baseline_mae:.4f}  (baseline)")
    results.append({"model": model_name, "experiment": "FULL",
                    "removed_group": "none", "n_features": len(feature_cols),
                    "mae": round(baseline_mae, 4), "delta_mae": 0.0,
                    "rel_delta_mae": 0.0, "note": ""})

    # ── Per-group ablation ────────────────────────────────────────────────────
    for group_name, group_cols in groups.items():
        present = [c for c in group_cols if c in feature_cols]

        # TFT: check if this group maps to TFT covariates
        if model_name == "tft":
            tft_mapping = TFT_GROUP_COVARIATES.get(group_name)
            if tft_mapping is None:
                # Group has no covariate equivalent in TFT's 4-covariate base
                note = TFT_GROUP_NOTES.get(group_name, f"N/A — {group_name} not in TFT covariate base")
                print(f"\n  [-{group_name}]  TFT N/A: {note}")
                results.append({
                    "model": model_name, "experiment": f"-{group_name}",
                    "removed_group": group_name,
                    "n_features": len(feature_cols),
                    "mae": float("nan"), "delta_mae": float("nan"),
                    "rel_delta_mae": float("nan"), "note": note
                })
                continue

        if not present and model_name != "tft":
            print(f"\n  [-{group_name}] SKIP — no columns present in feature set")
            continue

        reduced_cols = [c for c in feature_cols if c not in present]
        print(f"\n  [-{group_name}]  removing {len(present)} sklearn cols → "
              f"{len(reduced_cols)} remain")

        if model_name == "ridge":
            mae = eval_sklearn(df_feat, reduced_cols, target_cols, "ridge")
        elif model_name == "xgb":
            mae = eval_sklearn(df_feat, reduced_cols, target_cols, "xgb")
        elif model_name == "tft":
            mae = eval_tft(df_raw, res, lag_steps, n_targets, ablated_group=group_name)

        delta      = mae - baseline_mae
        rel_delta  = delta / baseline_mae if baseline_mae > 0 else 0
        impact     = "🔴 HIGH" if rel_delta > 0.15 else ("🟠 MED" if rel_delta > 0.05 else "🟢 LOW")
        extra_note = TFT_GROUP_NOTES.get(group_name, "") if model_name == "tft" else ""
        print(f"    MAE = {mae:.4f}  ΔMAE = {delta:+.4f}  rel={rel_delta:+.1%}  {impact}")
        if extra_note:
            print(f"    NOTE: {extra_note}")

        results.append({
            "model": model_name, "experiment": f"-{group_name}",
            "removed_group": group_name,
            "n_features": len(reduced_cols) if model_name != "tft" else len(feature_cols),
            "mae": round(mae, 4), "delta_mae": round(delta, 4),
            "rel_delta_mae": round(rel_delta, 4), "note": extra_note
        })

    return results


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--models", nargs="+",
                        choices=["ridge", "xgb", "tft", "timellm"],
                        default=["ridge", "xgb", "tft", "timellm"])
    args, _ = parser.parse_known_args()

    all_results = []

    for model_name in args.models:

        if model_name == "timellm":
            print(f"\n{'═'*65}")
            print("  TIMELLM — feature ablation NOT applicable")
            print("  TimeLLM only uses occupancy history; no exogenous inputs.")
            print(f"{'═'*65}")
            cfg    = BEST_CONFIGS["timellm"]
            df_raw = load_dataset(cfg["resolution_min"])
            _, _, _, lag_steps, n_targets = build_features(
                df_raw, cfg["resolution_min"], cfg["lag_hrs"])
            baseline = eval_timellm(df_raw, cfg["resolution_min"], lag_steps, n_targets)
            print(f"  Baseline MAE = {baseline:.4f}")
            all_results.append({
                "model": "timellm", "experiment": "FULL",
                "removed_group": "none", "n_features": "N/A",
                "mae": round(baseline, 4), "delta_mae": 0.0, "rel_delta_mae": 0.0,
                "note": "TimeLLM does not use exogenous features; ablation not applicable"
            })
            continue

        results = run_ablation_for_model(model_name)
        all_results.extend(results)

    # ── Save full results ──────────────────────────────────────────────────────
    df_out = pd.DataFrame(all_results)
    df_out.to_csv("ablation_results_fixed.csv", index=False)

    # ── Summary — Ridge + XGB + TFT only; G5 TFT N/A excluded from average ──
    df_rank = df_out[
        (df_out["experiment"] != "FULL") &
        (df_out["model"] != "timellm") &
        (df_out["note"] == "")  # exclude N/A rows
    ]
    summary = (df_rank.groupby("removed_group")["rel_delta_mae"]
               .mean().sort_values(ascending=False)
               .reset_index()
               .rename(columns={"rel_delta_mae": "avg_rel_delta_mae"}))
    summary["importance"] = summary["avg_rel_delta_mae"].apply(
        lambda x: "🔴 HIGH" if x > 0.15 else ("🟠 MEDIUM" if x > 0.05 else "🟢 LOW"))
    summary.to_csv("ablation_summary_fixed.csv", index=False)

    print(f"\n{'═'*65}")
    print("  ABLATION COMPLETE")
    print(f"{'═'*65}")
    print("\n  Feature Importance (avg relative ΔMAE — Ridge + XGB + TFT):")
    print("  Note: TimeLLM excluded. TFT G5_history shown separately (N/A).")
    print(summary.to_string(index=False))

    print(f"\n  Saved: ablation_results_fixed.csv, ablation_summary_fixed.csv")

    # Print per-model breakdown
    print(f"\n{'─'*65}")
    print("  Per-model breakdown:")
    print(df_out[["model", "experiment", "mae", "delta_mae",
                  "rel_delta_mae", "note"]].to_string(index=False))


if __name__ == "__main__":
    main()


═════════════════════════════════════════════════════════════════
  RIDGE  |  res=60min  lag=72h
═════════════════════════════════════════════════════════════════
  ✅  in_session: 105/704 rows active (14.9%)

  Feature groups resolved:
    G1_temporal: 4 cols  ['hour_of_day', 'day_of_week', 'is_weekend']...
    G2_schedule: 2 cols  ['in_session', 'fce_score']
    G3_capacity: 2 cols  ['capacity', 'extra_hours']
    G4_weather: 4 cols  ['temperature', 'precipitation', 'snowfall']...
    G5_history: 72 cols  ['occ_lag_1', 'occ_lag_2', 'occ_lag_3']...
    G6_future: 72 cols  ['in_session_f1', 'in_session_f2', 'in_session_f3']...

  [FULL] all features (157 cols)
    MAE = 1.8649  (baseline)

  [-G1_temporal]  removing 4 sklearn cols → 153 remain
    MAE = 1.8685  ΔMAE = +0.0035  rel=+0.2%  🟢 LOW

  [-G2_schedule]  removing 2 sklearn cols → 155 remain
    MAE = 1.8654  ΔMAE = +0.0005  rel=+0.0%  🟢 LOW

  [-G3_capacity]  removing 2 sklearn cols → 155 remain
    MAE = 1.8683  ΔMAE = +0.0033

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


    MAE = 1.8837  ΔMAE = +0.0636  rel=+3.5%  🟢 LOW

═════════════════════════════════════════════════════════════════
  TFT  |  res=30min  lag=48h
═════════════════════════════════════════════════════════════════
  ✅  in_session: 508/3156 rows active (16.1%)

  Feature groups resolved:
    G1_temporal: 4 cols  ['hour_of_day', 'day_of_week', 'is_weekend']...
    G2_schedule: 2 cols  ['in_session', 'fce_score']
    G3_capacity: 2 cols  ['capacity', 'extra_hours']
    G4_weather: 4 cols  ['temperature', 'precipitation', 'snowfall']...
    G5_history: 96 cols  ['occ_lag_1', 'occ_lag_2', 'occ_lag_3']...
    G6_future: 144 cols  ['in_session_f1', 'in_session_f2', 'in_session_f3']...

  [FULL] all features (253 cols)


INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=30` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

...

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

  Baseline MAE = 0.3385

═════════════════════════════════════════════════════════════════
  ABLATION COMPLETE
═════════════════════════════════════════════════════════════════

  Feature Importance (avg relative ΔMAE — Ridge + XGB + TFT):
  Note: TimeLLM excluded. TFT G5_history shown separately (N/A).
removed_group  avg_rel_delta_mae importance
   G5_history           1.802050     🔴 HIGH
  G1_temporal           1.536433     🔴 HIGH
  G2_schedule           0.975733     🔴 HIGH
    G6_future           0.239800     🔴 HIGH
  G3_capacity          -0.004400      🟢 LOW
   G4_weather          -0.025700      🟢 LOW

  Saved: ablation_results_fixed.csv, ablation_summary_fixed.csv

─────────────────────────────────────────────────────────────────
  Per-model breakdown:
  model   experiment    mae  delta_mae  rel_delta_mae                                                                             note
  ridge         FULL 1.8649     0.0000         0.0000                                            